# LCGP — Replicated 1D Illustration

This notebook demonstrates LCGP on 1D replicated data using three replication patterns:

- **Case 1**: Uniform-ish replication across the input space
- **Case 2**: Skewed replication (heavy in a subregion)
- **Case 3**: Hotspot replication (very high counts at specific locations)

Set `CASE` and `PLOT_MODE` in the configuration cell to switch between them.

## Imports

In [ ]:
import numpy as np
import tensorflow as tf
import time
import matplotlib.pyplot as plt
from pathlib import Path

from call_model import LCGPRun
from lcgp import evaluation

np.random.seed(42)
tf.random.set_seed(42)

## True Function

Three output dimensions derived from a common 1D input $x \in [0, 1]$.

In [ ]:
def f_true(x):
    x = np.asarray(x, dtype=np.float64)
    f1 = 0.8 + 0.3 * np.sin(2 * np.pi * x) + 0.2 * x
    f2 = 0.3 + 0.5 * np.cos(2 * np.pi * x)
    f3 = -0.4 - (x - 0.5) ** 2 + 0.2 * np.sin(4 * np.pi * x)
    return np.vstack([f1, f2, f3])

## Data Generation Functions

In [ ]:
def make_rep_data(n_unique=12, rep_choices=(1, 2, 3, 4), noise_std=(0.05, 0.08, 0.10), seed=None):
    """Case 1: Uniform-ish replication."""
    rng = np.random.default_rng(seed)
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)
    r = rng.choice(rep_choices, size=n_unique, replace=True)

    xs, ys = [], []
    for i, xi in enumerate(x_unique):
        yi_true = f_true([xi])[:, 0]
        for _ in range(int(r[i])):
            eps = rng.normal(0, noise_std, size=3).astype(np.float64)
            xs.append([xi])
            ys.append(yi_true + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue


def make_rep_data_skewed(n_unique=40, heavy_region=(0.20, 0.45),
                         light_rep_choices=(1, 2), heavy_rep_choices=(8, 12, 16, 20),
                         noise_std=(0.05, 0.08, 0.10), seed=None):
    """Case 2: Skewed replication — heavy in a contiguous subregion."""
    rng = np.random.default_rng(seed)
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)

    xs, ys = [], []
    for xi in x_unique:
        choices = heavy_rep_choices if (heavy_region[0] <= xi <= heavy_region[1]) else light_rep_choices
        r = int(rng.choice(choices))
        yi_base = f_true([xi])[:, 0]
        for _ in range(r):
            eps = np.array([rng.normal(0, s) for s in noise_std], dtype=np.float64)
            xs.append([xi])
            ys.append(yi_base + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue


def make_rep_data_hotspots(n_unique=50,
                           hotspots=((0.15, 10, 15), (0.50, 18, 25), (0.80, 12, 20)),
                           base_rep_choices=(1,), noise_std=(0.05, 0.08, 0.10), seed=None):
    """Case 3: Hotspot replication — very high counts at specific locations."""
    rng = np.random.default_rng(seed)
    x_unique = np.linspace(0.0, 1.0, n_unique, dtype=np.float64)
    hotspot_idx = {np.argmin(np.abs(x_unique - x0)): (lo, hi) for (x0, lo, hi) in hotspots}

    xs, ys = [], []
    for i, xi in enumerate(x_unique):
        if i in hotspot_idx:
            lo, hi = hotspot_idx[i]
            r = int(rng.integers(lo, hi + 1))
        else:
            r = int(rng.choice(base_rep_choices))
        yi_base = f_true([xi])[:, 0]
        for _ in range(r):
            eps = np.array([rng.normal(0, s) for s in noise_std], dtype=np.float64)
            xs.append([xi])
            ys.append(yi_base + eps)

    xtrain = np.array(xs, dtype=np.float64)
    ytrain = np.array(ys, dtype=np.float64).T
    xtest  = np.linspace(0.0, 1.0, 400, dtype=np.float64)[:, None]
    ytrue  = f_true(xtest[:, 0])
    return xtrain, ytrain, xtest, ytrue

## Configuration

Set `CASE` to `1`, `2`, or `3` and `PLOT_MODE` to `'y'` or `'g'`.

| `CASE` | Description |
|--------|-------------|
| `1` | Uniform-ish replication |
| `2` | Skewed replication (heavy in `[0.20, 0.45]`) |
| `3` | Hotspot replication |

| `PLOT_MODE` | Description |
|-------------|-------------|
| `'y'` | Plot outputs $f_1, f_2, f_3$ with replicates and credible bands |
| `'g'` | Plot latent GPs $g_1, g_2, g_3$ with training points |

In [ ]:
CASE      = 2       # 1, 2, or 3
SUBMETHOD = 'rep'   # 'rep' or 'full'
PLOT_MODE = 'y'     # 'y' or 'g'

## Generate Dataset

In [ ]:
if CASE == 1:
    results_fig_path = './results_figure_rep_1d_uniform/'
    xtrain, ytrain, xtest, ytrue = make_rep_data(
        n_unique=16, rep_choices=(1, 2, 3, 4, 5),
        noise_std=(0.05, 0.08, 0.10), seed=2025
    )
elif CASE == 2:
    results_fig_path = './results_figure_rep_1d_skewed/'
    xtrain, ytrain, xtest, ytrue = make_rep_data_skewed(
        n_unique=40, heavy_region=(0.20, 0.45),
        light_rep_choices=(1, 2), heavy_rep_choices=(8, 12, 16, 20),
        noise_std=(0.05, 0.08, 0.10), seed=123
    )
elif CASE == 3:
    results_fig_path = './results_figure_rep_1d_hotspots/'
    xtrain, ytrain, xtest, ytrue = make_rep_data_hotspots(
        n_unique=50, hotspots=((0.15, 10, 15), (0.50, 18, 25), (0.80, 12, 20)),
        base_rep_choices=(1,), noise_std=(0.05, 0.08, 0.10), seed=7
    )
else:
    raise ValueError('CASE must be 1, 2, or 3')

Path(results_fig_path).mkdir(parents=True, exist_ok=True)

print(f'Case {CASE} | N={xtrain.shape[0]} total obs | p={ytrain.shape[0]} outputs')

## Build and Train Model

In [ ]:
data = {
    'xtrain': xtrain,
    'xtest':  xtest,
    'ytrain': ytrain,
    'ytest':  ytrue,
    'ytrue':  ytrue
}

modelrun = LCGPRun(
    runno='rep_1d',
    data=data,
    num_latent=3,
    var_threshold=None,
    submethod=SUBMETHOD,
    diag_error_structure=[1, 1, 1],
    robust_mean=True,
)
modelrun.define_model()

t0 = time.time()
modelrun.train()
t1 = time.time()

print(f'Training time: {t1 - t0:.2f}s')

## Predict

In [ ]:
predmean, ypredvar, yconfvar = modelrun.predict(return_fullcov=False)

## Diagnostics

In [ ]:
mdl = modelrun.model

print('=== BASIS ===')
print(f'diag_D values: {mdl.diag_D.numpy()}')
print(f'phi^T @ phi diagonal: {np.diag(mdl.phi.numpy().T @ mdl.phi.numpy())}')

print('\n=== FITTED PARAMETERS ===')
lLmb, lLmb0, lsigma2s, lnugGPs = mdl.get_param()
for k in range(lLmb.shape[0]):
    print(f'  Lengthscale component {k}: {lLmb[k].numpy()}')
print(f'Noise log-var (lsigma2s): {lsigma2s.numpy()}')
print(f'Noise std (fitted): {np.sqrt(np.exp(lsigma2s.numpy()))}')
print(f'Noise std (true):   [0.05, 0.08, 0.10]')

print('\n=== REPLICATION STATS ===')
r = mdl.r.numpy()
print(f'Unique locations n: {len(r)}')
print(f'Total samples N:    {np.sum(r)}')
print(f'Avg / Min / Max reps: {np.mean(r):.2f} / {np.min(r)} / {np.max(r)}')

## Evaluation Metrics

In [ ]:
rmse   = evaluation.rmse(ytrue, predmean)
nrmse  = evaluation.normalized_rmse(ytrue, predmean)
pcover, pwidth = evaluation.intervalstats(ytrue, predmean, yconfvar)
dss    = evaluation.dss(ytrue, predmean, yconfvar, use_diag=True)

print(f'RMSE:            {rmse:.4f}')
print(f'NRMSE:           {nrmse:.4f}')
print(f'95% PI coverage: {pcover:.3f}')
print(f'95% PI width:    {pwidth:.4f}')
print(f'DSS:             {dss:.4f}')

## Plot Results

In [ ]:
order_test  = np.argsort(xtest[:, 0])
order_train = np.argsort(xtrain[:, 0])

if PLOT_MODE == 'g':
    _ = mdl.predict(x0=xtest, return_fullcov=False)
    ghat_test = np.asarray(mdl.ghat)
    gstd_test = np.sqrt(np.asarray(mdl.gvar))
    ghat_tr   = np.asarray(mdl.mks)
    x_tr_unique = np.asarray(mdl.x_unique)
    order_unique = np.argsort(x_tr_unique[:, 0])

    q = ghat_test.shape[0]
    fig, axes = plt.subplots(q, 1, figsize=(10, 1.9 * q), sharex=True)
    if q == 1:
        axes = [axes]

    x_test_sorted = xtest[order_test, 0]
    x_tr_sorted   = x_tr_unique[order_unique, 0]

    for k, ax in enumerate(axes):
        m = ghat_test[k, order_test]
        s = gstd_test[k, order_test]
        ax.plot(x_test_sorted, m, lw=1.8, label=fr'$g_{{{k+1}}}(x)$ mean')
        ax.fill_between(x_test_sorted, m - 1.96 * s, m + 1.96 * s,
                        alpha=0.22, label='95% band')
        ax.scatter(x_tr_sorted, ghat_tr[k, order_unique],
                   s=12, alpha=0.65, label='train pts')
        ax.set_ylabel(fr'$g_{{{k+1}}}(x)$')
        ax.legend(loc='best', fontsize=9)

    axes[-1].set_xlabel('x')
    plt.suptitle(f'Latent GPs — Case {CASE}', y=1.01)
    plt.tight_layout()
    outfile = Path(results_fig_path) / 'lcgp_latents_gkx.png'

else:  # PLOT_MODE == 'y'
    fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
    for i in range(3):
        ax[i].scatter(xtrain[order_train, 0], ytrain[i, order_train],
                      s=12, alpha=0.65,
                      label='replicates' if i == 0 else None)
        ax[i].plot(xtest[order_test, 0], ytrue[i, order_test],
                   lw=1.8, label='true' if i == 0 else None)
        ax[i].plot(xtest[order_test, 0], predmean[i, order_test],
                   lw=1.5, label='LCGP mean' if i == 0 else None)
        lo = predmean[i, order_test] - 1.96 * np.sqrt(yconfvar[i, order_test])
        hi = predmean[i, order_test] + 1.96 * np.sqrt(yconfvar[i, order_test])
        ax[i].fill_between(xtest[order_test, 0], lo, hi,
                           alpha=0.22,
                           label='95% credible band' if i == 0 else None)
        ax[i].set_ylabel(f'$f_{i+1}(x)$')

    ax[-1].set_xlabel('x')
    ax[0].legend(loc='best', fontsize=9)
    plt.suptitle(f'LCGP Predictions — Case {CASE}', y=1.01)
    plt.tight_layout()
    outfile = Path(results_fig_path) / 'lcgp_rep_1d_demo.png'

plt.savefig(outfile, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {outfile}')